In [1]:
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

In [2]:
def find_repository_root(start_path=Path.cwd()):
    for folder in [start_path, *start_path.parents]:
        if (folder / ".git").exists():
            return folder
    
    raise FileNotFoundError(
        "Could not locate the Git repository."
    )


REPO_ROOT = find_repository_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)

Repository: D:\analytics\A_Python_Code\creditlens-rag
Raw data: D:\analytics\A_Python_Code\creditlens-rag\data\raw
Processed data: D:\analytics\A_Python_Code\creditlens-rag\data\processed


In [3]:
html_files = sorted(RAW_DATA_DIR.glob("*.html"))

print("Number of HTML filings:", len(html_files))

for file_path in html_files:
    print(file_path.name)

Number of HTML filings: 9
F_2023_10K.html
F_2024_10K.html
F_2025_10K.html
GM_2023_10K.html
GM_2024_10K.html
GM_2025_10K.html
TSLA_2023_10K.html
TSLA_2024_10K.html
TSLA_2025_10K.html


In [4]:
def clean_sec_html(file_path, remove_tables=True):
    """
    Extract readable narrative text from an SEC HTML filing.
    """
    
    html = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )
    
    soup = BeautifulSoup(html, "lxml")
    
    unwanted_tags = [
        "script",
        "style",
        "noscript",
        "svg"
    ]
    
    if remove_tables:
        unwanted_tags.append("table")
    
    for tag in soup.find_all(unwanted_tags):
        tag.decompose()
    
    text = soup.get_text(separator=" ")
    
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    
    return text

In [5]:
sample_file = RAW_DATA_DIR / "F_2025_10K.html"

ford_text = clean_sec_html(sample_file)

print("Characters:", len(ford_text))
print("Words:", len(ford_text.split()))

C:\Users\veln8\AppData\Local\Temp\ipykernel_20424\2389262001.py:11: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Characters: 651552
Words: 86356


In [6]:
print(ford_text[:3000])

f-20251231 FALSE 2025 FY 0000037996 http://fasb.org/us-gaap/2025#AccountsPayableAndAccruedLiabilitiesCurrent http://fasb.org/us-gaap/2025#AccountsPayableAndAccruedLiabilitiesCurrent http://fasb.org/us-gaap/2025#CostOfGoodsAndServicesSold http://fasb.org/us-gaap/2025#AccountsReceivableNetCurrent http://fasb.org/us-gaap/2025#CostOfGoodsAndServicesSold http://fasb.org/us-gaap/2025#CostOfGoodsAndServicesSold P2Y 1 1 .3333 .3333 .3333 http://fasb.org/us-gaap/2025#OtherAssets http://fasb.org/us-gaap/2025#OtherAssets http://fasb.org/us-gaap/2025#RevenueFromContractWithCustomerExcludingAssessedTax http://fasb.org/us-gaap/2025#RevenueFromContractWithCustomerExcludingAssessedTax http://fasb.org/us-gaap/2025#RevenueFromContractWithCustomerExcludingAssessedTax http://fasb.org/us-gaap/2025#IncomeLossFromEquityMethodInvestments http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.

In [7]:
for keyword in [
    "risk factors",
    "liquidity",
    "indebtedness",
    "interest rates"
]:
    print(
        keyword,
        "→",
        keyword.lower() in ford_text.lower()
    )

risk factors → True
liquidity → True
indebtedness → True
interest rates → True


In [9]:
metadata_path = REPO_ROOT / "data" / "filings_metadata.csv"

filings_metadata = pd.read_csv(
    metadata_path,
    dtype={"cik": str}
)

filings_metadata["reporting_year"] = (
    filings_metadata["reportDate"]
    .astype(str)
    .str[:4]
)

filings_metadata[
    [
        "company",
        "ticker",
        "reporting_year",
        "filing_url"
    ]
]

,company,ticker,reporting_year,filing_url
0,Ford,F,2025,https://www.sec.gov/Archives/edgar/data/37996/...
1,Ford,F,2024,https://www.sec.gov/Archives/edgar/data/37996/...
2,Ford,F,2023,https://www.sec.gov/Archives/edgar/data/37996/...
3,General Motors,GM,2025,https://www.sec.gov/Archives/edgar/data/146785...
4,General Motors,GM,2024,https://www.sec.gov/Archives/edgar/data/146785...
5,General Motors,GM,2023,https://www.sec.gov/Archives/edgar/data/146785...
6,Tesla,TSLA,2025,https://www.sec.gov/Archives/edgar/data/131860...
7,Tesla,TSLA,2024,https://www.sec.gov/Archives/edgar/data/131860...
8,Tesla,TSLA,2023,https://www.sec.gov/Archives/edgar/data/131860...


In [10]:
metadata_lookup = {
    (row["ticker"], row["reporting_year"]): row
    for _, row in filings_metadata.iterrows()
}

print("Metadata records:", len(metadata_lookup))

Metadata records: 9


In [11]:
documents = []

for file_path in html_files:
    
    # Example: F_2025_10K.html
    file_parts = file_path.stem.split("_")
    
    ticker = file_parts[0]
    reporting_year = file_parts[1]
    
    metadata = metadata_lookup[
        (ticker, reporting_year)
    ]
    
    cleaned_text = clean_sec_html(
        file_path,
        remove_tables=True
    )
    
    document = {
        "document_id": f"{ticker}_{reporting_year}_10K",
        "company": metadata["company"],
        "ticker": ticker,
        "reporting_year": reporting_year,
        "form": "10-K",
        "filing_date": metadata["filingDate"],
        "source_file": file_path.name,
        "source_url": metadata["filing_url"],
        "character_count": len(cleaned_text),
        "word_count": len(cleaned_text.split()),
        "text": cleaned_text
    }
    
    documents.append(document)
    
    print(
        f"Processed {file_path.name}: "
        f"{document['word_count']:,} words"
    )

C:\Users\veln8\AppData\Local\Temp\ipykernel_20424\2389262001.py:11: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Processed F_2023_10K.html: 83,493 words
Processed F_2024_10K.html: 85,279 words
Processed F_2025_10K.html: 86,356 words
Processed GM_2023_10K.html: 56,456 words
Processed GM_2024_10K.html: 58,391 words
Processed GM_2025_10K.html: 58,745 words
Processed TSLA_2023_10K.html: 52,166 words
Processed TSLA_2024_10K.html: 50,645 words
Processed TSLA_2025_10K.html: 54,990 words


In [12]:
documents_df = pd.DataFrame(documents)

document_summary = documents_df.drop(
    columns=["text"]
)

document_summary

,document_id,company,ticker,reporting_year,form,filing_date,source_file,source_url,character_count,word_count
0,F_2023_10K,Ford,F,2023,10-K,2024-02-07,F_2023_10K.html,https://www.sec.gov/Archives/edgar/data/37996/...,651373,83493
1,F_2024_10K,Ford,F,2024,10-K,2025-02-06,F_2024_10K.html,https://www.sec.gov/Archives/edgar/data/37996/...,660666,85279
2,F_2025_10K,Ford,F,2025,10-K,2026-02-11,F_2025_10K.html,https://www.sec.gov/Archives/edgar/data/37996/...,651552,86356
3,GM_2023_10K,General Motors,GM,2023,10-K,2024-01-30,GM_2023_10K.html,https://www.sec.gov/Archives/edgar/data/146785...,415138,56456
4,GM_2024_10K,General Motors,GM,2024,10-K,2025-01-28,GM_2024_10K.html,https://www.sec.gov/Archives/edgar/data/146785...,427308,58391
5,GM_2025_10K,General Motors,GM,2025,10-K,2026-01-27,GM_2025_10K.html,https://www.sec.gov/Archives/edgar/data/146785...,431474,58745
6,TSLA_2023_10K,Tesla,TSLA,2023,10-K,2024-01-29,TSLA_2023_10K.html,https://www.sec.gov/Archives/edgar/data/131860...,364017,52166
7,TSLA_2024_10K,Tesla,TSLA,2024,10-K,2025-01-30,TSLA_2024_10K.html,https://www.sec.gov/Archives/edgar/data/131860...,354775,50645
8,TSLA_2025_10K,Tesla,TSLA,2025,10-K,2026-01-29,TSLA_2025_10K.html,https://www.sec.gov/Archives/edgar/data/131860...,388080,54990


In [13]:
print("Documents processed:", len(documents_df))
print("Total words:", f"{documents_df['word_count'].sum():,}")

Documents processed: 9
Total words: 586,521


In [14]:
documents_df[
    [
        "document_id",
        "company",
        "word_count",
        "character_count"
    ]
].sort_values("word_count")

,document_id,company,word_count,character_count
7,TSLA_2024_10K,Tesla,50645,354775
6,TSLA_2023_10K,Tesla,52166,364017
8,TSLA_2025_10K,Tesla,54990,388080
3,GM_2023_10K,General Motors,56456,415138
4,GM_2024_10K,General Motors,58391,427308
5,GM_2025_10K,General Motors,58745,431474
0,F_2023_10K,Ford,83493,651373
1,F_2024_10K,Ford,85279,660666
2,F_2025_10K,Ford,86356,651552


In [15]:
clean_documents_path = (
    PROCESSED_DATA_DIR / "clean_documents.jsonl"
)

documents_df.to_json(
    clean_documents_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved to:", clean_documents_path)
print(
    "File size:",
    round(
        clean_documents_path.stat().st_size / (1024 * 1024),
        2
    ),
    "MB"
)

Saved to: D:\analytics\A_Python_Code\creditlens-rag\data\processed\clean_documents.jsonl
File size: 4.16 MB
